In [33]:
# 1-1
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_1.csv')
# print(df.info())
# print(df.head())

df1 = df.groupby('소주제')['정답여부'].sum() / df.groupby('소주제')['정답여부'].count()
df2 = df1.sort_values(ascending=False)
df2.rank(method='dense', ascending=False)

print(df1['경제'])

0.68


In [22]:
# 1-2
import numpy as np
import pandas as pd
df = pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_2.csv')
# print(df.info())
# print(df.head())

df['year'] = df['date'].str.split('-').str[0]
df['month'] = df['date'].str.split('-').str[1]

df1 = df.groupby(['year', 'month'])['price'].sum().sort_values(ascending=False)
# print(df1.index[3])

print(df[(df['year'] ==  '2024') & (df['month'] ==  '12')].groupby('category')['price'].sum().sort_values(ascending=False)[0])

1012500


C:\Users\user\AppData\Local\Temp\ipykernel_10672\1942741096.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(df[(df['year'] ==  '2024') & (df['month'] ==  '12')].groupby('category')['price'].sum().sort_values(ascending=False)[0])


In [44]:
# 1-3
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_1_3.csv')
# print(df.info())
# print(df.head())

df['split'] = df['message'].str.split(' ')
df['split'] = df['split'].apply(lambda x: len(x))
print(np.abs(np.round(df.groupby('label')['split'].mean()[0] - df.groupby('label')['split'].mean()[1], 3)))

0.3


C:\Users\user\AppData\Local\Temp\ipykernel_10672\2309175908.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(np.abs(np.round(df.groupby('label')['split'].mean()[0] - df.groupby('label')['split'].mean()[1], 3)))
C:\Users\user\AppData\Local\Temp\ipykernel_10672\2309175908.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(np.abs(np.round(df.groupby('label')['split'].mean()[0] - df.groupby('label')['split'].mean()[1], 3)))


In [3]:
# 2
import numpy as np
import pandas as pd
train_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_train.csv')
test_df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_2_test.csv')

print(train_df.info())
print(train_df.head())
# print(test_df.info())

# gas_totl에 결측치를 0으로 대체해놨다고 했기때문에 0이라는 값이 y를 예측하는데 도움이 되지 않을 수 있음 그래서 0인 행을 제거함.
train_df = train_df[train_df['gas_totl'] != 0]

train_y = train_df['gas_totl']
train_X = train_df.drop(columns=['gas_totl'])
test_y = test_df['gas_totl']
test_X = test_df.drop(columns=['gas_totl'])

from sklearn.model_selection import train_test_split
train_X, valid_X, train_y, valid_y = train_test_split(train_X, train_y, test_size=0.3)
print(train_X.shape, train_y.shape, valid_X.shape, valid_y.shape)

train_X_cat = train_X.select_dtypes('object').columns.tolist()
train_X_num = train_X.select_dtypes('number').columns.tolist()
valid_X_cat = valid_X.select_dtypes('object').columns.tolist()
valid_X_num = valid_X.select_dtypes('number').columns.tolist()
test_X_cat = test_X.select_dtypes('object').columns.tolist()
test_X_num = test_X.select_dtypes('number').columns.tolist()

from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

train_X_encoder = encoder.fit_transform(train_X[train_X_cat])
valid_X_encoder = encoder.transform(valid_X[valid_X_cat])
test_X_encoder = encoder.transform(test_X[test_X_cat])

train_X = np.concatenate([train_X[train_X_num], train_X_encoder], axis=1)
valid_X = np.concatenate([valid_X[valid_X_num], valid_X_encoder], axis=1)
test_X = np.concatenate([test_X[test_X_num], test_X_encoder], axis=1)

from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor()
rf.fit(train_X, train_y)
pred2 = rf.predict(train_X)
print(root_mean_squared_error(train_y, pred2))

from sklearn.metrics import root_mean_squared_error
pred = rf.predict(valid_X)
print(root_mean_squared_error(valid_y, pred))

pred1 = rf.predict(test_X)
# print(root_mean_squared_error(test_y, pred1))

pd.DataFrame(pred1, columns=['pred']).to_csv('result.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 160 entries, 0 to 159
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   biz_type        160 non-null    object 
 1   area            160 non-null    float64
 2   age             160 non-null    int64  
 3   num_households  160 non-null    int64  
 4   gas_totl        160 non-null    int64  
dtypes: float64(1), int64(3), object(1)
memory usage: 6.4+ KB
None
      biz_type       area  age  num_households  gas_totl
0       public  93.291780   33              61       499
1  residential  97.509495   34             131      1156
2  residential  97.140181    5              30       392
3  residential  65.789408   12             166      1343
4        mixed  92.631000   28              14       154
(98, 4) (98,) (42, 4) (42,)
26.307236530611696
76.07771615419388


In [94]:
# 3-1
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_3_1.csv')
# print(df.info())
# print(df.head())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model = smf.glm('attrition ~ age + income + overtime', data=df, family=sm.families.Binomial()).fit()
model.pvalues < 0.05

print(np.round(model.params['income'],3))
print(np.round(np.exp(model.params['age']),3))
a = pd.DataFrame({'age': [20], 'income': [3000], 'overtime': [2]})
print(np.round(model.predict(a)[0], 3))

-0.005
0.905
0.431


In [124]:
# 3-2
import numpy as np
import pandas as pd
df=pd.read_csv('https://raw.githubusercontent.com/YoungjinBD/data/main/exam/10_3_2.csv')
# print(df.info())
# print(df.head())

import statsmodels.api as sm
import statsmodels.formula.api as smf
model = smf.ols('price ~ area + height + wall', data=df).fit()

df1 = model.pvalues[1:] < 0.05
a = df1[df1==True].index.tolist()
# print(a)
print(model.params[a].sum().round(3))

model1 = smf.ols('price ~ area + height', data=df).fit()
print(model1.rsquared.round(3))

b = pd.DataFrame({'area': [100], 'height': [10], 'wall': [1]})
print(model1.predict(b).values[0].round(3))

10.289
0.859
329.036
